In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.append("../utils")
sys.path.append("../training")
import matplotlib.pyplot as plt
import matplotlib
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from geometric_features import get_path_sig
import iisignature
from stage_dataset import StageDataset
from tqdm.auto import tqdm

tqdm.pandas()

# Ideas:
* Do different stages form coherent clusters via path sigs? (this should be fairly fruitful)
* Do different grades form coherent clusters via path sigs? (probably not but could be cool to show different shapes)
* Are there any other patterns, say for certain kinds of clusters, images look one way. (might be a rather involved analysis)

In [2]:
# hyperparameters
model_name = "convlstm2-2026-07-09" #"convlstm_final-2026-07-13"
TIME_OFFSET = 0.3
PCA_DIM = 8

In [3]:
# load latents and metadata
metadata_df = pd.read_csv(os.path.join("latents",f"{model_name}.csv"))
latents = np.load(os.path.join("latents",f"{model_name}.npy"))
latents_df = pd.DataFrame(latents, columns=[f"z_{i}" for i in range(latents.shape[1])], index=metadata_df.index)
pca_cols = [f"pca_{i}" for i in range(PCA_DIM)]
pca_latents = PCA(n_components=PCA_DIM).fit_transform(StandardScaler().fit_transform(latents))
pca_latents_df = pd.DataFrame(pca_latents, columns=pca_cols, index=metadata_df.index)
df = pd.concat([metadata_df, latents_df, pca_latents_df], axis=1)

In [ ]:
ps_cols = [f"path_sig_{i}" for i in range(len(iisignature.basis(iisignature.prepare(PCA_DIM+1, 3))))]

def path_sig_agg(group):
    pca_traj = group[pca_cols].to_numpy()
    path_sig = get_path_sig(pca_traj, 3, time_offsets=TIME_OFFSET)
    out_df = pd.DataFrame(path_sig[None, :], columns = ps_cols)
    out_df["phase"] = [group.name[1]] # the second key is 'phase'
    return out_df
path_sig_df = df.groupby(["embryo_id", "phase"]).progress_apply(path_sig_agg).reset_index(drop=True)
visual_ps = PCA(n_components=3).fit_transform(StandardScaler().fit_transform(path_sig_df[ps_cols].to_numpy()))
visual_ps_colors = [StageDataset.PHASES[p] for p in path_sig_df['phase'].to_list()]


  0%|          | 0/9422 [00:00<?, ?it/s]

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(8,6), subplot_kw={"projection":"3d"})
ax.scatter(visual_ps[:,0], visual_ps[:,1], visual_ps[:,2], c=visual_ps_colors, cmap='tab20c', vmin=0, vmax=19)
plt.show()
plt.close(fig)